<a href="XXXX" target="_parent"><img src="XXXX" alt="Open In Colab"/></a>

## Libraries

In [1]:
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount = True)
    IN_COLAB = True
except:
    IN_COLAB = False

In [2]:
if IN_COLAB:
  !pip install transformers
  !pip install datasets
  !pip install evaluate
  !pip install sentencepiece

In [3]:
import os
import torch

if IN_COLAB:
    root_path = 'Enter drive path'
else:
    root_path = '/root/InstructABSA-EB712'
    
use_mps = True if torch.has_mps else False
os.chdir(root_path)

/tmp/ipykernel_10319/419096211.py:9: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  use_mps = True if torch.has_mps else False


In [4]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

from InstructABSA.data_prep import DatasetLoader
from InstructABSA.utils import T5Generator, T5Classifier
from instructions import InstructionsHandler

## Training

In [5]:
task_name = 'joint_task'
experiment_name = 'lapt2014_iabsa1'
model_checkpoint = 'allenai/tk-instruct-base-def-pos'
print('Experiment Name: ', experiment_name)
model_out_path = './Models'
model_out_path = os.path.join(model_out_path, task_name, f"{model_checkpoint.replace('/', '')}-{experiment_name}")
print('Model output path: ', model_out_path)

Experiment Name:  lapt2014_iabsa1
Model output path:  ./Models/joint_task/allenaitk-instruct-base-def-pos-lapt2014_iabsa1


In [6]:
# Load the data
id_train_file_path = './Dataset/SemEval14/Train/Laptops_Train.csv'
id_test_file_path = './Dataset/SemEval14/Test/Laptops_Test.csv'
id_tr_df = pd.read_csv(id_train_file_path)
id_te_df = pd.read_csv(id_test_file_path)

# Get the input text into the required format using Instructions
instruct_handler = InstructionsHandler()

# Set instruction_set1 for InstructABSA-1 and instruction_set2 for InstructABSA-2
instruct_handler.load_instruction_set1()

# Set bos_instruct1 for lapt14 and bos_instruct2 for rest14. For other datasets, modify the insructions.py file.
loader = DatasetLoader(id_tr_df, id_te_df)

if loader.train_df_id is not None:
    loader.train_df_id = loader.create_data_in_aspe_format(
        loader.train_df_id, 'term', 'polarity', 'raw_text', 'aspectTerms', 
        instruct_handler.aspe['bos_instruct1'], instruct_handler.aspe['eos_instruct']
    )
if loader.test_df_id is not None:
    loader.test_df_id = loader.create_data_in_aspe_format(
        loader.test_df_id, 'term', 'polarity', 'raw_text', 'aspectTerms', 
        instruct_handler.aspe['bos_instruct1'], instruct_handler.aspe['eos_instruct']
    )

In [7]:
# After loading and formatting the data
from sklearn.model_selection import train_test_split

if loader.train_df_id is not None:
    # Split training data to create a validation set
    train_df, val_df = train_test_split(loader.train_df_id, test_size=0.2, random_state=42)
    
    # Update the loader with the split data
    loader.train_df_id = train_df
    loader.val_df_id = val_df  # Set validation data
    
    print(f"Training set size: {len(train_df)} examples")
    print(f"Validation set size: {len(val_df)} examples")

Training set size: 2436 examples
Validation set size: 609 examples


In [8]:
# Create T5 utils object
t5_exp = T5Generator(model_checkpoint)

In [9]:

# Tokenize Dataset
id_ds, id_tokenized_ds, ood_ds, ood_tokenized_ds = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

# Training arguments
training_args = {
    'output_dir': model_out_path,
    'evaluation_strategy': "epoch",
    'learning_rate': 5e-5,
    'lr_scheduler_type': 'cosine',
    'per_device_train_batch_size': 8,
    'per_device_eval_batch_size': 16,
    'num_train_epochs': 4,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'save_strategy': 'epoch',  # Save at each epoch
    'load_best_model_at_end': True,  # Load the best model at the end
    'metric_for_best_model': 'eval_loss',  # Use validation loss as metric
    'greater_is_better': False,  # Lower loss is better
    'push_to_hub': False,
    'eval_accumulation_steps': 1,
    'predict_with_generate': True,
    'use_mps_device': use_mps
}

Map: 100%|██████████| 609/609 [00:00<00:00, 5276.55 examples/s]


In [10]:
# Train model
model_trainer = t5_exp.train(id_tokenized_ds, **training_args)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Trainer device: cuda:0

Model training started ....


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,No log,0.243641
2,0.456600,0.235443
3,0.456600,0.229615
4,0.216700,0.222370


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


## Inference

In [ ]:
# Load the data
id_train_file_path = './Dataset/SemEval14/Train/Laptops_Train.csv'
id_test_file_path = './Dataset/SemEval14/Test/Laptops_Test.csv'
id_tr_df = pd.read_csv(id_train_file_path)
id_te_df = pd.read_csv(id_test_file_path)

# Get the input text into the required format using Instructions
instruct_handler = InstructionsHandler()

# Set instruction_set1 for InstructABSA-1 and instruction_set2 for InstructABSA-2
instruct_handler.load_instruction_set1()

# Set bos_instruct1 for lapt14 and bos_instruct2 for rest14. For other datasets, modify the insructions.py file.
loader = DatasetLoader(id_tr_df, id_te_df)# Load the data 
id_train_file_path = './Dataset/SemEval14/Train/Laptops_Train.csv'
id_test_file_path = './Dataset/SemEval14/Test/Laptops_Test.csv'
id_tr_df = pd.read_csv(id_train_file_path)
id_te_df = pd.read_csv(id_test_file_path)

# Get the input text into the required format using Instructions
instruct_handler = InstructionsHandler()

# Set instruction_set1 for InstructABSA-1 and instruction_set2 for InstructABSA-2
instruct_handler.load_instruction_set1()

# Set bos_instruct1 for lapt14 and bos_instruct2 for rest14. For other datasets, modify the insructions.py file.
loader = DatasetLoader(id_tr_df, id_te_df)

if loader.train_df_id is not None:
    loader.train_df_id = loader.create_data_in_aspe_format(
        loader.train_df_id, 'term', 'polarity', 'raw_text', 'aspectTerms', 
        instruct_handler.aspe['bos_instruct1'], instruct_handler.aspe['eos_instruct']
    )
if loader.test_df_id is not None:
    loader.test_df_id = loader.create_data_in_aspe_format(
        loader.test_df_id, 'term', 'polarity', 'raw_text', 'aspectTerms', 
        instruct_handler.aspe['bos_instruct1'], instruct_handler.aspe['eos_instruct']
    )

# Split training data to create a validation set
from sklearn.model_selection import train_test_split

if loader.train_df_id is not None:
    # Split training data into train and validation sets
    train_df, val_df = train_test_split(loader.train_df_id, test_size=0.2, random_state=42)
    
    # Update the loader with the split data
    loader.train_df_id = train_df
    loader.val_df_id = val_df
    
    print(f"Training set size: {len(train_df)} examples")
    print(f"Validation set size: {len(val_df)} examples")
if loader.train_df_id is not None:
    loader.train_df_id = loader.create_data_in_joint_task_format(loader.train_df_id, 'term', 'polarity', 'raw_text', 'aspectTerms', instruct_handler.joint['bos_instruct1'], instruct_handler.joint['eos_instruct'])
if loader.test_df_id is not None:
    loader.test_df_id = loader.create_data_in_joint_task_format(loader.test_df_id, 'term', 'polarity', 'raw_text', 'aspectTerms', instruct_handler.joint['bos_instruct1'], instruct_handler.joint['eos_instruct'])

Training set size: 2436 examples
Validation set size: 609 examples


In [13]:
# Model inference - Loading from Checkpoint
t5_exp = T5Generator(model_out_path)

# Tokenize Datasets
id_ds, id_tokenized_ds, ood_ds, ood_tokenzed_ds = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

# Get prediction labels - Training set   
id_tr_pred_labels = t5_exp.get_labels(tokenized_dataset = id_tokenized_ds, sample_set = 'train', batch_size = 16)
id_tr_labels = [i.strip() for i in id_ds['train']['labels']]

# Get prediction labels - Testing set
id_te_pred_labels = t5_exp.get_labels(tokenized_dataset = id_tokenized_ds, sample_set = 'test', batch_size = 16)
id_te_labels = [i.strip() for i in id_ds['test']['labels']]

Map: 100%|██████████| 609/609 [00:00<00:00, 4931.23 examples/s]


Model loaded to:  cuda


100%|██████████| 153/153 [00:40<00:00,  3.78it/s]


Model loaded to:  cuda


100%|██████████| 50/50 [00:13<00:00,  3.65it/s]


In [14]:
p, r, f1, _ = t5_exp.get_metrics(id_tr_labels, id_tr_pred_labels)
print('Train Precision: ', p)
print('Train Recall: ', r)
print('Train F1: ', f1)

p, r, f1, _ = t5_exp.get_metrics(id_te_labels, id_te_pred_labels)
print('Test Precision: ', p)
print('Test Recall: ', r)
print('Test F1: ', f1)

Train Precision:  0.827866190321533
Train Recall:  0.8128188775510204
Train F1:  0.8202735317779567
Test Precision:  0.7936191425722832
Test Recall:  0.7713178294573644
Test F1:  0.7823095823095824
